In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account
import os
from dotenv import load_dotenv

load_dotenv()

# Configuración
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Cliente autenticado
credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH
)
client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

# Crear dataset
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "EU"  # Datos en Europa
dataset = client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset {DATASET_ID} creado")

Dataset jointech creado


In [3]:
schema = [
    bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("first_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("last_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("email", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("phone", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("country", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("city", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("acquisition_channel", "STRING", mode="NULLABLE"),  # organic, paid_ads, social_media, referral, email, other
    bigquery.SchemaField("registered_at", "DATETIME", mode="REQUIRED"),
]

table_ref = f"{PROJECT_ID}.{DATASET_ID}.customers"
table = bigquery.Table(table_ref, schema=schema)
table = client.create_table(table, exists_ok=True)
print("Tabla customers creada")

Tabla customers creada


In [4]:
schema_categories = [
    bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("active", "BOOL", mode="REQUIRED"),
    bigquery.SchemaField("created_at", "DATETIME", mode="REQUIRED"),
]

table_ref = f"{PROJECT_ID}.{DATASET_ID}.categories"
table = bigquery.Table(table_ref, schema=schema_categories)
table = client.create_table(table, exists_ok=True)
print("Tabla categories creada")


schema_products = [
    bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),   # FK → categories
    bigquery.SchemaField("sku", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("sale_price", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("cost_price", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("stock_quantity", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("active", "BOOL", mode="REQUIRED"),
    bigquery.SchemaField("created_at", "DATETIME", mode="REQUIRED"),
    bigquery.SchemaField("updated_at", "DATETIME", mode="NULLABLE"),
]

table_ref = f"{PROJECT_ID}.{DATASET_ID}.products"
table = bigquery.Table(table_ref, schema=schema_products)
table = client.create_table(table, exists_ok=True)
print("Tabla products creada")


schema_orders = [
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),   # FK → customers
    bigquery.SchemaField("order_status", "STRING", mode="REQUIRED"), # pending, confirmed, shipped, delivered, cancelled, returned
    bigquery.SchemaField("shipping_address", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("shipping_city", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("shipping_country", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("shipping_postal_code", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("shipping_cost", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("ordered_at", "DATETIME", mode="REQUIRED"),
    bigquery.SchemaField("shipped_at", "DATETIME", mode="NULLABLE"),
    bigquery.SchemaField("delivered_at", "DATETIME", mode="NULLABLE"),
    bigquery.SchemaField("cancelled_at", "DATETIME", mode="NULLABLE"),
]

table_ref = f"{PROJECT_ID}.{DATASET_ID}.orders"
table = bigquery.Table(table_ref, schema=schema_orders)
table = client.create_table(table, exists_ok=True)
print("Tabla orders creada")


schema_order_items = [
    bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),      # FK → orders
    bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),    # FK → products
    bigquery.SchemaField("quantity", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("unit_price_at_purchase", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("unit_cost_at_purchase", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("discount_amount", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("line_total", "NUMERIC", mode="REQUIRED"),
]

table_ref = f"{PROJECT_ID}.{DATASET_ID}.order_items"
table = bigquery.Table(table_ref, schema=schema_order_items)
table = client.create_table(table, exists_ok=True)
print("Tabla order_items creada")


schema_payments = [
    bigquery.SchemaField("payment_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),      # FK → orders
    bigquery.SchemaField("payment_method", "STRING", mode="REQUIRED"), # card, paypal, bank_transfer, cash_on_delivery, other
    bigquery.SchemaField("payment_status", "STRING", mode="REQUIRED"), # completed, refunded, pending, failed
    bigquery.SchemaField("amount", "NUMERIC", mode="REQUIRED"),
    bigquery.SchemaField("paid_at", "DATETIME", mode="NULLABLE"),
    bigquery.SchemaField("transaction_reference", "STRING", mode="NULLABLE"),
]

table_ref = f"{PROJECT_ID}.{DATASET_ID}.payments"
table = bigquery.Table(table_ref, schema=schema_payments)
table = client.create_table(table, exists_ok=True)
print("Tabla payments creada")


schema_reviews = [
    bigquery.SchemaField("review_id", "INT64", mode="REQUIRED"),
    bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"), # FK → order_items
    bigquery.SchemaField("rating", "INT64", mode="REQUIRED"),        # Valor entre 1 y 5
    bigquery.SchemaField("comment", "STRING", mode="NULLABLE"),
    bigquery.SchemaField("reviewed_at", "DATETIME", mode="REQUIRED"),
]

table_ref = f"{PROJECT_ID}.{DATASET_ID}.reviews"
table = bigquery.Table(table_ref, schema=schema_reviews)
table = client.create_table(table, exists_ok=True)
print("Tabla reviews creada")

Tabla categories creada
Tabla products creada
Tabla orders creada
Tabla order_items creada
Tabla payments creada
Tabla reviews creada
